In [1]:
# ==========================================================
# STUDENT 3
# NOTEBOOK 02
# BERT + XGBOOST TRAINING
# ==========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import joblib

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

warnings.filterwarnings("ignore")

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

EMBEDDINGS = PROJECT_ROOT / "embeddings"

MODELS = PROJECT_ROOT / "models"

REPORTS = PROJECT_ROOT / "reports"

FIGURES = PROJECT_ROOT / "figures"

BERT_EMBEDDINGS = EMBEDDINGS / "bert"

print("="*70)
print("BERT + XGBOOST TRAINING")
print("="*70)

BERT + XGBOOST TRAINING


In [3]:
# ==========================================================
# LOAD BERT EMBEDDINGS
# ==========================================================

X_train = np.load(
    BERT_EMBEDDINGS / "train_embeddings.npy"
)

X_validation = np.load(
    BERT_EMBEDDINGS / "validation_embeddings.npy"
)

y_train = np.load(
    BERT_EMBEDDINGS / "train_labels.npy"
)

y_validation = np.load(
    BERT_EMBEDDINGS / "validation_labels.npy"
)

print("="*70)
print("EMBEDDINGS LOADED")
print("="*70)

print("Training Features :", X_train.shape)
print("Validation Features :", X_validation.shape)

print("Training Labels :", y_train.shape)
print("Validation Labels :", y_validation.shape)

EMBEDDINGS LOADED
Training Features : (5000, 768)
Validation Features : (1000, 768)
Training Labels : (5000,)
Validation Labels : (1000,)


In [4]:
# ==========================================================
# VERIFY DATA
# ==========================================================

print("="*70)
print("TRAIN LABEL DISTRIBUTION")
print("="*70)

print(pd.Series(y_train).value_counts())

print()

print("="*70)
print("VALIDATION LABEL DISTRIBUTION")
print("="*70)

print(pd.Series(y_validation).value_counts())

TRAIN LABEL DISTRIBUTION
0    4552
1     448
Name: count, dtype: int64

VALIDATION LABEL DISTRIBUTION
0    910
1     90
Name: count, dtype: int64


In [5]:
# ==========================================================
# BASELINE XGBOOST
# ==========================================================

baseline_model = XGBClassifier(

    objective="binary:logistic",

    eval_metric="logloss",

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    n_jobs=-1

)

print("="*70)
print("TRAINING BASELINE XGBOOST")
print("="*70)

baseline_model.fit(

    X_train,

    y_train

)

print("="*70)
print("BASELINE TRAINING COMPLETED")
print("="*70)

TRAINING BASELINE XGBOOST
BASELINE TRAINING COMPLETED


In [6]:
# ==========================================================
# VALIDATION PREDICTIONS
# ==========================================================

validation_predictions = baseline_model.predict(

    X_validation

)

validation_probabilities = baseline_model.predict_proba(

    X_validation

)[:,1]

accuracy = accuracy_score(

    y_validation,

    validation_predictions

)

print("="*70)
print("VALIDATION ACCURACY")
print("="*70)

print(f"{accuracy:.4f}")

VALIDATION ACCURACY
0.9930


In [8]:
# ==========================================================
# HYPERPARAMETER OPTIMIZATION
# ==========================================================

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [9]:
# ==========================================================
# XGBOOST SEARCH SPACE
# ==========================================================

parameter_space = {

    "n_estimators": randint(200, 500),

    "max_depth": randint(3, 8),

    "learning_rate": uniform(0.02, 0.18),

    "subsample": uniform(0.70, 0.30),

    "colsample_bytree": uniform(0.70, 0.30),

    "min_child_weight": randint(1, 6),

    "gamma": uniform(0.0, 0.4)

}

print("="*70)
print("SEARCH SPACE CREATED")
print("="*70)

SEARCH SPACE CREATED


In [10]:
# ==========================================================
# RANDOM SEARCH
# ==========================================================

random_search = RandomizedSearchCV(

    estimator=XGBClassifier(

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=42,

        n_jobs=-1

    ),

    param_distributions=parameter_space,

    n_iter=30,

    scoring="f1",

    cv=3,

    random_state=42,

    verbose=2,

    n_jobs=-1,

    refit=True

)

print("="*70)
print("RANDOM SEARCH READY")
print("="*70)

RANDOM SEARCH READY


In [11]:
# ==========================================================
# START OPTIMIZATION
# ==========================================================

print("="*70)
print("STARTING RANDOM SEARCH")
print("="*70)

random_search.fit(

    X_train,

    y_train

)

print("="*70)
print("OPTIMIZATION FINISHED")
print("="*70)

print("Best Parameters")

print(random_search.best_params_)

print()

print("Best Cross Validation F1")

print(random_search.best_score_)

STARTING RANDOM SEARCH
Fitting 3 folds for each of 30 candidates, totalling 90 fits
OPTIMIZATION FINISHED
Best Parameters
{'colsample_bytree': np.float64(0.8123620356542087), 'gamma': np.float64(0.3802857225639665), 'learning_rate': np.float64(0.1517589095260529), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 302, 'subsample': np.float64(0.8337498258560773)}

Best Cross Validation F1
0.9932884399551066


In [12]:
# ==========================================================
# SAVE BEST MODEL
# ==========================================================

best_xgb = random_search.best_estimator_

joblib.dump(

    best_xgb,

    MODELS /

    "best_bert_xgboost.pkl"

)

print("="*70)
print("BEST BERT-XGBOOST MODEL SAVED")
print("="*70)

BEST BERT-XGBOOST MODEL SAVED


In [13]:
# ==========================================================
# SAVE BEST PARAMETERS
# ==========================================================

best_parameters = pd.DataFrame(

    [random_search.best_params_]

)

best_parameters["Best_F1"] = random_search.best_score_

display(best_parameters)

best_parameters.to_csv(

    REPORTS /

    "bert_xgboost_best_parameters.csv",

    index=False

)

print("="*70)
print("BEST PARAMETERS SAVED")
print("="*70)

,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,subsample,Best_F1
0,0.812362,0.380286,0.151759,7,5,302,0.83375,0.993288


BEST PARAMETERS SAVED
